In [2]:
import pandas as pd
import random
import math
df = pd.read_csv('../lessons/lesson_9/accommodation.csv')

In [ ]:
# считаем руками, без sklearn

quality_values = df["OverallQual"].tolist()
area_values    = df["GrLivArea"].tolist()
price_values   = df["SalePrice"].tolist()

# ЦЕЛЬ: дорогой дом (1) или нет (0) — дороже медианы
median_price = sorted(price_values)[len(price_values)//2]
label_values = []
for price in price_values:
    if price > median_price:
        label_values.append(1)
    else:
        label_values.append(0)

def select(values, indices):
    return [values[i] for i in indices]
def mean(values):
    return sum(values) / len(values)
def std_dev(values):
    m = mean(values)
    return (sum((v - m)**2 for v in values) / len(values))**0.5
def standardize(values, average, spread):
    return [(v - average) / spread for v in values]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

# делим на train / test
random.seed(42)
indices = list(range(len(label_values)))
random.shuffle(indices)
cut = int(len(indices) * 0.8)
train_i, test_i = indices[:cut], indices[cut:]

quality_train, quality_test = select(quality_values, train_i), select(quality_values, test_i)
area_train,    area_test    = select(area_values, train_i),    select(area_values, test_i)
label_train,   label_test   = select(label_values, train_i),   select(label_values, test_i)

# нормализуем признаки (mean/std из train)
quality_mean, quality_std = mean(quality_train), std_dev(quality_train)
area_mean,    area_std    = mean(area_train),    std_dev(area_train)
quality_train_norm = standardize(quality_train, quality_mean, quality_std)
quality_test_norm  = standardize(quality_test,  quality_mean, quality_std)
area_train_norm    = standardize(area_train,    area_mean,    area_std)
area_test_norm     = standardize(area_test,     area_mean,    area_std)

# ОБУЧАЕМ
weight_quality = 0.0
weight_area    = 0.0
bias           = 0.0
learning_rate  = 0.1
num_samples    = len(label_train)

for step in range(3000):
    grad_quality = 0.0
    grad_area    = 0.0
    grad_bias    = 0.0
    for quality, area, actual in zip(quality_train_norm, area_train_norm, label_train):
        z = weight_quality*quality + weight_area*area + bias
        probability = sigmoid(z)                 # предсказание = вероятность
        error = probability - actual             # та же ошибка, что в линейной!
        grad_quality += error * quality
        grad_area    += error * area
        grad_bias    += error
    weight_quality -= learning_rate * grad_quality / num_samples
    weight_area    -= learning_rate * grad_area / num_samples
    bias           -= learning_rate * grad_bias / num_samples

# ОЦЕНИВАЕМ (accuracy)
def predict_probability(quality, area):
    return sigmoid(weight_quality*quality + weight_area*area + bias)

correct = 0
for quality, area, actual in zip(quality_test_norm, area_test_norm, label_test):
    probability = predict_probability(quality, area)
    predicted_class = 1 if probability > 0.5 else 0
    if predicted_class == actual:
        correct += 1
accuracy = correct / len(label_test)

print("веса:", round(weight_quality,3), round(weight_area,3), " bias:", round(bias,3))
print("accuracy на тесте:", round(accuracy, 3))

веса: 2.238 1.527  bias: 0.097
accuracy на тесте: 0.853


In [5]:
# коробочное решение

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

X = df[["OverallQual", "GrLivArea"]]
median_price = df["SalePrice"].median()
y = (df["SalePrice"] > median_price).astype(int)     # 1 = дорогой, 0 = нет

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm  = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_norm, y_train)

predictions   = model.predict(X_test_norm)             # жёсткие классы 0/1
probabilities = model.predict_proba(X_test_norm)[:, 1] # вероятности класса 1

print("accuracy: ", accuracy_score(y_test, predictions))
print("precision:", precision_score(y_test, predictions))
print("recall:   ", recall_score(y_test, predictions))
print("f1:       ", f1_score(y_test, predictions))
print("roc-auc:  ", roc_auc_score(y_test, probabilities))   # тут вероятности!

accuracy:  0.8835616438356164
precision: 0.875968992248062
recall:    0.8625954198473282
f1:        0.8692307692307693
roc-auc:   0.955739414916315
